In [1]:
import sys
sys.path.append("..")

import pandas as pd

from src.embeddings import NILCEmbeddings, tokenizar
from src.vectorstore import carregar_colecao
from src.retrieval import processar_pergunta
from src.perguntas_teste import PERGUNTAS

embeddings = NILCEmbeddings()
store = carregar_colecao(embeddings)
print("Registros no Chroma:", store._collection.count())

linhas = []
for tipo, esperado, pergunta in PERGUNTAS:
    consulta = processar_pergunta(pergunta)
    conhecidas = [t for t in tokenizar(consulta) if t in embeddings.vocab]

    if not conhecidas:  # nenhuma palavra no vocabulário NILC: não dá para gerar vetor
        linhas.append(dict(tipo=tipo, esperado=esperado, pergunta=pergunta,
                           palavras_conhecidas=0, sim_top1=None, sim_media_top5=None))
        continue

    sims = [1 - d for _, d in store.similarity_search_with_score(consulta, k=10)]
    linhas.append(dict(tipo=tipo, esperado=esperado, pergunta=pergunta,
                       palavras_conhecidas=len(conhecidas),
                       sim_top1=round(sims[0], 3),
                       sim_media_top5=round(sum(sims[:5]) / 5, 3)))

tabela = pd.DataFrame(linhas)
pd.set_option("display.max_colwidth", 70)
print(tabela.sort_values(["esperado", "sim_top1"], ascending=[True, False]).to_string(index=False))

Registros no Chroma: 40393
              tipo  esperado                                                               pergunta  palavras_conhecidas  sim_top1  sim_media_top5
      quantitativa   recusar                        Quantas avaliações mencionam atraso na entrega?                    5     0.825           0.818
especifica_ausente   recusar      Há reclamações sobre a assinatura mensal do serviço de streaming?                    7     0.801           0.790
especifica_ausente   recusar                       Os clientes relatam vazamento de dados pessoais?                    5     0.771           0.760
      quantitativa   recusar                                      Qual é a nota média dos clientes?                    3     0.760           0.756
       fora_escopo   recusar                                   Qual será a cotação do dólar amanhã?                    4     0.673           0.670
       fora_escopo   recusar                     Qual foi o desempenho financeiro da Olist 

In [2]:
print(tabela.groupby("esperado")[["sim_top1", "sim_media_top5"]].describe().T)

esperado               recusar  responder
sim_top1       count  8.000000   8.000000
               mean   0.682625   0.827625
               std    0.127933   0.038925
               min    0.477000   0.763000
               25%    0.592000   0.798250
               50%    0.716500   0.837000
               75%    0.778500   0.850000
               max    0.825000   0.877000
sim_media_top5 count  8.000000   8.000000
               mean   0.671750   0.816000
               std    0.130783   0.036971
               min    0.466000   0.760000
               25%    0.567000   0.784250
               50%    0.713000   0.825500
               75%    0.767500   0.836750
               max    0.818000   0.861000


In [3]:
base = pd.read_parquet("../data/processed/reviews_indexed.parquet")
for termo in ["vazamento|vazou|dados pessoais", "streaming"]:
    n = base["texto"].str.contains(termo, case=False, regex=True).sum()
    print(f"{termo!r}: {n} avaliações")

'vazamento|vazou|dados pessoais': 20 avaliações
'streaming': 0 avaliações


perguntas fora do escopo de resposta ou invalidas estao trazerndo evidencias como 'kkkkk', assim como perguntas de vazamento de dados, vamos verificar se os comentarios da base possuem dados pessoais ou se sao apenas comentarios de produtos com liquido vazado e etc.

In [4]:
base = pd.read_parquet("../data/processed/reviews_indexed.parquet")

dp = base[base["texto"].str.contains("dados pessoais", case=False)]
vaz = base[base["texto"].str.contains("vazamento|vazou", case=False)]
print("dados pessoais:", len(dp))
print("vazamento/vazou:", len(vaz))
print(vaz["texto"].sample(min(8, len(vaz)), random_state=1).to_string())

dados pessoais: 0
vazamento/vazou: 20
3506     A caixa veio amassada e molhada. Veio quatro frasco de 500 ML e um...
35187    O produto veio mais rápido do que esperava porém o frasco veio fal...
15101    o produto veio com os frascos dos elementos químicos vazios. Não s...
23297    O produto precisa ser melhor vedado para evitar vazamentos.Mas de ...
2865     Fiz o teste assim q recebi e amei, oliquido nao vazou da capa e fo...
28763    Quem embalou devia ter colocado o produto em pé e não deitado, poi...
6912     Meu produto é 2 bateria de 12V 70A foi entregue mas vazou ácido na...
39779    Recomendo a loja, porém devem tomar cuidado ao fazer a postagem de...


testando o piso de bloqueio para perguntas legitimas.

adicionamos mais perguntas no conjunto de perguntas teste para verificar se o piso de bloqueio ira recusar perguntas legitimas
    ("curta", "responder", "Entrega atrasada?"),
    ("curta", "responder", "Produtos quebrados"),
    ("curta", "responder", "Atendimento ruim"),
    ("curta", "responder", "Qualidade dos produtos"),